<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.1-B — EXTENDED HIGH-R SINGLE-REPLICATION LEARNING CURVE
#
# R = 20,000, 30,000, 50,000, 100,000
#
# IMPORTANT DESIGN PRINCIPLE
# --------------------------
# The original 50,000 training configurations are PRESERVED EXACTLY.
# An additional independent 50,000-design is appended.
#
# Hence:
#
#   D_20k subset D_30k subset D_50k subset D_100k
#
# and the previously completed R<=50,000 results remain scientifically comparable.
#
# PERSISTENCE
# -----------
# - Google Drive permanent storage
# - exact targets saved in chunks
# - previous compatible 50k exact targets reused
# - previous R=20k/30k/50k models/results reused when verified
# - neural training checkpoint every 50 mini-batches
# - epoch-end checkpoint
# - optimizer / scheduler / RNG states saved
# - completed R results permanently saved
#
# AFTER COLAB DISCONNECT
# ----------------------
# Reconnect and run THIS SAME CELL.
#
# LOSS
# ----
# L = L_PMF + lambda_rho L_tail + lambda_tau L_tau
#
# SPARSE LU ONLY — NO MATRIX INVERSE
# =====================================================================================


# =====================================================================================
# 0. GOOGLE DRIVE
# =====================================================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# =====================================================================================
# 1. IMPORTS
# =====================================================================================

from __future__ import annotations

import os
import copy
import math
import pickle
import random
import shutil
import time

from dataclasses import dataclass, asdict
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

from joblib import Parallel, delayed

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# =====================================================================================
# 2. CONFIG
# =====================================================================================

@dataclass
class Config:

    seed:int = 20260820

    beta_range:Tuple[float,float] = (.30,1.50)
    gamma_range:Tuple[float,float] = (.20,1.00)
    omega_range:Tuple[float,float] = (.02,.50)

    initial_fraction_range:Tuple[float,float] = (.02,.20)
    i0_one_fraction:float = .25

    train_N:Tuple[int,...] = (
        30,50,70,90,110,130,150
    )

    interp_N:Tuple[int,...] = (
        40,60,80,100,120,140
    )

    # -------------------------------------------------------------
    # EXTENDED TRAINING SET
    # -------------------------------------------------------------
    n_train:int = 100000

    n_val:int = 500

    n_test_seen:int = 500
    n_test_interp:int = 450

    width:int = 128
    depth:int = 3
    batch_size:int = 64

    epochs:int = 500

    lr:float = 1e-3
    weight_decay:float = 1e-6

    lambda_rho:float = 1.0
    lambda_tau:float = .02

    patience:int = 20
    min_delta:float = 1e-6
    grad_clip:float = 5.

    prob_tol:float = 1e-10
    var_rel_tol:float = 1e-8
    kl_eps:float = 1e-12
    refine_steps:int = 3

    teacher_version:str = (
        "B_R100000_N150_tailaware_single_rep_v16"
    )

    output_dir:str = (
        "results_section_5_1B_R100000_single_rep"
    )

    dpi:int = 300


cfg = Config()


# =====================================================================================
# 3. EXPERIMENT SETTINGS
# =====================================================================================

R_VALUES = (
    20000,
    30000,
    50000,
    100000
)


# Gallery only at R values that are actually trained / available.
GALLERY_R = R_VALUES


N_BASE = 50000
N_EXTRA = 50000


assert cfg.n_train == (
    N_BASE + N_EXTRA
)

assert max(R_VALUES) == (
    cfg.n_train
)


EXACT_TEST_N = (
    cfg.train_N
)

INTERP_TEST_N = (
    cfg.interp_N
)


assert set(
    EXACT_TEST_N
).isdisjoint(
    INTERP_TEST_N
)


N_scale = max(
    cfg.train_N
)


# =====================================================================================
# 4. PERMANENT GOOGLE-DRIVE DIRECTORIES
# =====================================================================================

ROOT = Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_1B_R100000_resumable_v16"
)


# Previous persistent 50k experiment.
OLD_ROOT = Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_1B_R50000_resumable_v15"
)


CACHE = ROOT/"cache"

EXACT_CACHE = CACHE/"exact"
MODEL_CACHE = CACHE/"models"
CHECKPOINT_CACHE = CACHE/"training_checkpoints"

OUT = ROOT/"results"


for d in (
    ROOT,
    CACHE,
    EXACT_CACHE,
    MODEL_CACHE,
    CHECKPOINT_CACHE,
    OUT
):

    d.mkdir(
        parents=True,
        exist_ok=True
    )


OLD_EXACT_CACHE = (
    OLD_ROOT
    /
    "cache"
    /
    "exact"
)


OLD_MODEL_CACHE = (
    OLD_ROOT
    /
    "cache"
    /
    "models"
)


OLD_PROGRESS_FILE = (
    OLD_ROOT
    /
    "learning_curve_progress.pkl"
)


print("="*115)
print("EXPERIMENT 5.1-B — R UP TO 100,000")
print("="*115)

print(
    "Permanent root:",
    ROOT
)

print(
    "Previous 50k root:",
    OLD_ROOT
)

print(
    "R values:",
    R_VALUES
)

print(
    "Base configurations:",
    f"{N_BASE:,}"
)

print(
    "Additional configurations:",
    f"{N_EXTRA:,}"
)

print(
    "Total configurations:",
    f"{cfg.n_train:,}"
)


# =====================================================================================
# 5. ATOMIC SAVING
# =====================================================================================

def atomic_pickle(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        tmp,
        "wb"
    ) as f:

        pickle.dump(
            obj,
            f,
            pickle.HIGHEST_PROTOCOL
        )

    os.replace(
        tmp,
        path
    )


def atomic_torch_save(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        obj,
        tmp
    )

    os.replace(
        tmp,
        path
    )


def safe_pickle_load(
    path,
    default=None
):

    try:

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(
                f
            )

    except Exception:

        return default


# =====================================================================================
# 6. REPRODUCIBILITY / DEVICE
# =====================================================================================

def seed_all(s):

    random.seed(
        s
    )

    np.random.seed(
        s
    )

    torch.manual_seed(
        s
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            s
        )


seed_all(
    cfg.seed
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


CPU = (
    os.cpu_count()
    or
    1
)


# Conservative exact parallelism for Colab stability.
N_EXACT = max(
    1,
    min(
        2,
        CPU
    )
)


if device.type == "cpu":

    torch.set_num_threads(
        max(
            1,
            min(
                8,
                CPU
            )
        )
    )


try:

    torch.set_num_interop_threads(
        1
    )

except RuntimeError:

    pass


print(
    "Device:",
    device
)

print(
    "Logical CPU cores:",
    CPU
)

print(
    "Exact workers:",
    N_EXACT
)


# =====================================================================================
# 7. SIRS TOPOLOGY
# =====================================================================================

@lru_cache(maxsize=None)
def topology(N):

    states = [

        (s,i)

        for i in range(
            1,
            N+1
        )

        for s in range(
            N-i+1
        )

    ]


    idx = {

        x:j

        for j,x
        in enumerate(states)

    }


    M = len(
        states
    )


    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]


    db = np.zeros(M)
    dg = np.zeros(M)
    dw = np.zeros(M)
    qb = np.zeros(M)


    for row,(s,i) in enumerate(
        states
    ):

        r = (
            N-s-i
        )


        # Infection
        if s:

            ir.append(
                row
            )

            ic.append(
                idx[
                    (s-1,i+1)
                ]
            )

            rate = (
                s*i/N
            )

            ib.append(
                rate
            )

            db[row] = (
                rate
            )


        # Recovery
        dg[row] = (
            i
        )


        if i==1:

            qb[row] = (
                i
            )

        else:

            rr.append(
                row
            )

            rc.append(
                idx[
                    (s,i-1)
                ]
            )

            rb.append(
                i
            )


        # Immunity loss
        if r:

            wr.append(
                row
            )

            wc.append(
                idx[
                    (s+1,i)
                ]
            )

            wb.append(
                r
            )

            dw[row] = (
                r
            )


    return (

        idx,
        M,

        np.asarray(
            ir,
            dtype=int
        ),

        np.asarray(
            ic,
            dtype=int
        ),

        np.asarray(
            ib,
            dtype=float
        ),

        np.asarray(
            rr,
            dtype=int
        ),

        np.asarray(
            rc,
            dtype=int
        ),

        np.asarray(
            rb,
            dtype=float
        ),

        np.asarray(
            wr,
            dtype=int
        ),

        np.asarray(
            wc,
            dtype=int
        ),

        np.asarray(
            wb,
            dtype=float
        ),

        db,
        dg,
        dw,
        qb
    )


# =====================================================================================
# 8. GENERATOR MATRICES
# =====================================================================================

def matrices(
    beta,
    gamma,
    omega,
    N
):

    (
        idx,
        M,

        ir,
        ic,
        ib,

        rr,
        rc,
        rb,

        wr,
        wc,
        wb,

        db,
        dg,
        dw,
        qb

    ) = topology(
        N
    )


    rows = np.concatenate(
        [
            ir,
            rr,
            wr,
            np.arange(M)
        ]
    )


    cols = np.concatenate(
        [
            ic,
            rc,
            wc,
            np.arange(M)
        ]
    )


    vals = np.concatenate(
        [

            beta*ib,

            gamma*rb,

            omega*wb,

            -(
                beta*db
                +
                gamma*dg
                +
                omega*dw
            )

        ]
    )


    T = sparse.coo_matrix(

        (
            vals,
            (
                rows,
                cols
            )
        ),

        shape=(
            M,
            M
        ),

        dtype=np.float64

    ).tocsc()


    D1 = sparse.coo_matrix(

        (
            beta*ib,
            (
                ir,
                ic
            )
        ),

        shape=(
            M,
            M
        ),

        dtype=np.float64

    ).tocsc()


    q = (
        gamma*qb
    )


    return (

        T,

        (
            T-D1
        ).tocsc(),

        D1,

        q,

        idx

    )


# =====================================================================================
# 9. SPARSE-LU SOLVES
# =====================================================================================

def factor(
    A,
    ordering="COLAMD"
):

    return splu(
        A.tocsc(),
        permc_spec=ordering
    )


def solve_lu(
    A,
    lu,
    b,
    transpose=False
):

    b = np.asarray(
        b,
        dtype=np.float64
    )


    mode = (
        "T"
        if transpose
        else
        "N"
    )


    x = lu.solve(
        b,
        trans=mode
    )


    for _ in range(
        cfg.refine_steps
    ):

        residual = (

            b-A.T@x

            if transpose

            else b-A@x

        )


        if not np.all(
            np.isfinite(
                residual
            )
        ):

            break


        rel = (

            np.linalg.norm(
                residual,
                np.inf
            )

            /

            max(
                np.linalg.norm(
                    b,
                    np.inf
                ),
                1.
            )

        )


        if rel<1e-11:

            break


        x += lu.solve(
            residual,
            trans=mode
        )


    return np.asarray(
        x,
        dtype=np.float64
    )


def moment_factor(
    A,
    ordering
):

    d = np.abs(
        A.diagonal()
    )


    scale = (

        1.

        /

        np.maximum(
            d,
            np.finfo(float).tiny
        )

    )


    As = (

        sparse.diags(
            scale
        )

        @

        A

    ).tocsc()


    lu = splu(
        As,
        permc_spec=ordering
    )


    return (
        lu,
        scale
    )


def moment_solve(
    A,
    lu,
    scale,
    b
):

    b = np.asarray(
        b,
        dtype=np.float64
    )


    x = lu.solve(
        scale*b
    )


    for _ in range(
        cfg.refine_steps
    ):

        residual = (
            b-A@x
        )


        if not np.all(
            np.isfinite(
                residual
            )
        ):

            break


        rel = (

            np.linalg.norm(
                residual,
                np.inf
            )

            /

            max(
                np.linalg.norm(
                    b,
                    np.inf
                ),
                1.
            )

        )


        if rel<1e-11:

            break


        x += lu.solve(
            scale*residual
        )


    return np.asarray(
        x,
        dtype=np.float64
    )


# =====================================================================================
# 10. EXTINCTION MOMENTS
# =====================================================================================

def variance_linear_system(
    T,
    q,
    A,
    lu,
    scale,
    m1
):

    C = T.tocoo()


    keep = (
        C.row
        !=
        C.col
    )


    rr = C.row[
        keep
    ]


    cc = C.col[
        keep
    ]


    rates = C.data[
        keep
    ]


    source = np.bincount(

        rr,

        weights=(

            rates

            *

            (
                m1[cc]
                -
                m1[rr]
            )**2

        ),

        minlength=T.shape[0]

    ).astype(
        np.float64
    )


    source += (
        q*m1*m1
    )


    if (
        not np.all(
            np.isfinite(
                source
            )
        )
        or
        source.min()<-1e-8
    ):

        raise ArithmeticError(
            "Invalid variance-system RHS."
        )


    return moment_solve(

        A,
        lu,
        scale,

        np.maximum(
            source,
            0.
        )

    )


def extinction_moments(
    T,
    q,
    initial
):

    A = (
        -T
    ).tocsc()


    one = np.ones(
        A.shape[0],
        dtype=np.float64
    )


    errors = []


    for ordering in (
        "COLAMD",
        "MMD_AT_PLUS_A"
    ):

        try:

            lu,scale = moment_factor(
                A,
                ordering
            )


            # (-T) m1 = 1
            m1 = moment_solve(
                A,
                lu,
                scale,
                one
            )


            mean = float(
                m1[
                    initial
                ]
            )


            if (
                not np.all(
                    np.isfinite(
                        m1
                    )
                )
                or
                mean<=0
            ):

                raise ArithmeticError(
                    f"Invalid E(tau)={mean}"
                )


            # (-T) m2 = 2m1
            m2 = moment_solve(
                A,
                lu,
                scale,
                2.*m1
            )


            second = float(
                m2[
                    initial
                ]
            )


            if (
                not np.all(
                    np.isfinite(
                        m2
                    )
                )
                or
                second<=0
            ):

                raise ArithmeticError(
                    f"Invalid E(tau^2)={second}"
                )


            raw = (

                np.longdouble(
                    second
                )

                -

                np.longdouble(
                    mean
                )**2

            )


            tol = (

                cfg.var_rel_tol

                *

                max(
                    abs(second),
                    mean*mean,
                    1.
                )

            )


            if (
                np.isfinite(raw)
                and
                raw>=-tol
            ):

                var = max(
                    float(raw),
                    0.
                )

                method = (
                    "moment subtraction"
                )


            else:

                vv = variance_linear_system(

                    T,
                    q,
                    A,
                    lu,
                    scale,
                    m1

                )


                var = float(
                    vv[
                        initial
                    ]
                )


                method = (
                    "variance linear system"
                )


            if (
                not np.isfinite(
                    var
                )
                or
                var<0
            ):

                raise ArithmeticError(
                    f"Invalid Var(tau)={var}"
                )


            return (

                mean,
                second,
                var,
                True,

                f"{ordering}; {method}"

            )


        except Exception as e:

            errors.append(
                f"{ordering}: {e}"
            )


    return (

        np.nan,
        np.nan,
        np.nan,
        False,

        " | ".join(
            errors
        )

    )


# =====================================================================================
# 11. EXACT TEACHER
# =====================================================================================

def exact_targets(
    beta,
    gamma,
    omega,
    N,
    i0
):

    (
        T,
        D0,
        D1,
        q,
        idx

    ) = matrices(

        beta,
        gamma,
        omega,
        N

    )


    M = (
        T.shape[0]
    )


    initial = idx[
        (
            N-i0,
            i0
        )
    ]


    # -------------------------------------------------------------------------
    # Infection-count distribution
    # -------------------------------------------------------------------------

    A0 = (
        -D0
    ).tocsc()


    lu0 = factor(
        A0
    )


    alpha = np.zeros(
        M
    )


    alpha[
        initial
    ] = 1.


    b = solve_lu(
        A0,
        lu0,
        q
    )


    v = (
        alpha.copy()
    )


    p = np.zeros(
        N+2,
        dtype=np.float64
    )


    for k in range(
        N+1
    ):

        p[k] = (
            v@b
        )


        y = solve_lu(

            A0,
            lu0,
            v,

            transpose=True

        )


        v = np.asarray(
            D1.T@y
        ).ravel()


    p[-1] = (
        v.sum()
    )


    p[
        np.abs(p)<cfg.prob_tol
    ] = 0.


    if (
        not np.all(
            np.isfinite(
                p
            )
        )
        or
        p.min()<-cfg.prob_tol
    ):

        raise RuntimeError(
            f"Invalid exact PMF: "
            f"N={N}, i0={i0}"
        )


    p = np.maximum(
        p,
        0.
    )


    mass = (
        p.sum()
    )


    if (
        not np.isfinite(
            mass
        )
        or
        abs(mass-1.)>1e-5
    ):

        raise RuntimeError(
            f"Invalid PMF mass={mass}: "
            f"N={N}, i0={i0}"
        )


    p /= (
        mass
    )


    (
        mean,
        second,
        var,
        tau_valid,
        tau_info

    ) = extinction_moments(
        T,
        q,
        initial
    )


    return (

        p,
        mean,
        second,
        var,
        tau_valid,
        tau_info

    )


# =====================================================================================
# 12. DATA RECORD
# =====================================================================================

@dataclass
class Record:

    beta:float
    gamma:float
    omega:float

    N:int
    i0:int

    p:np.ndarray

    mean_tau:float
    second_tau:float
    var_tau:float

    tau_valid:bool
    tau_info:str


# =====================================================================================
# 13. EXPERIMENTAL DESIGN
# =====================================================================================

def stretch(
    u,
    bounds
):

    a,b = bounds

    return (
        a
        +
        (b-a)*u
    )


def design(
    n,
    Ns,
    seed
):

    U = qmc.LatinHypercube(
        d=4,
        seed=seed
    ).random(
        n
    )


    beta = stretch(
        U[:,0],
        cfg.beta_range
    )


    gamma = stretch(
        U[:,1],
        cfg.gamma_range
    )


    omega = stretch(
        U[:,2],
        cfg.omega_range
    )


    frac = stretch(
        U[:,3],
        cfg.initial_fraction_range
    )


    Nv = np.tile(

        np.asarray(
            Ns
        ),

        math.ceil(
            n/len(Ns)
        )

    )[:n]


    rng = np.random.default_rng(
        seed+991
    )


    rng.shuffle(
        Nv
    )


    i0 = np.asarray([

        int(
            np.clip(
                round(
                    frac[j]*Nv[j]
                ),
                2,
                Nv[j]
            )
        )

        for j in range(n)

    ])


    for N in Ns:

        ix = np.where(
            Nv==N
        )[0]


        if len(ix)==0:

            continue


        k = max(

            1,

            int(
                round(
                    cfg.i0_one_fraction
                    *
                    len(ix)
                )
            )

        )


        i0[
            rng.choice(
                ix,
                k,
                replace=False
            )
        ] = 1


    return [

        (

            float(
                beta[j]
            ),

            float(
                gamma[j]
            ),

            float(
                omega[j]
            ),

            int(
                Nv[j]
            ),

            int(
                i0[j]
            )

        )

        for j in range(n)

    ]


# =====================================================================================
# 14. DATASET COMPATIBILITY CHECK
#
# Used before reusing any previous R<=50k target file.
# =====================================================================================

def records_match_configs(
    records,
    configs,
    tol=1e-12
):

    if records is None:

        return False


    if len(records)!=len(configs):

        return False


    for r,x in zip(
        records,
        configs
    ):

        b,g,w,N,i0 = (
            x
        )


        if (
            abs(r.beta-b)>tol
            or
            abs(r.gamma-g)>tol
            or
            abs(r.omega-w)>tol
            or
            r.N!=N
            or
            r.i0!=i0
        ):

            return False


    return True


# =====================================================================================
# 15. RESUMABLE EXACT-TARGET GENERATION
# =====================================================================================

EXACT_CHUNK = 200


def _exact_record(
    local_index,
    x
):

    (
        b,
        g,
        w,
        N,
        i0

    ) = x


    try:

        (
            p,
            m,
            m2,
            v,
            ok,
            info

        ) = exact_targets(
            b,
            g,
            w,
            N,
            i0
        )


    except Exception as e:

        raise RuntimeError(

            f"Exact teacher failed | "
            f"N={N}, i0={i0}, "
            f"beta={b:.8g}, "
            f"gamma={g:.8g}, "
            f"omega={w:.8g}\n"
            f"{e}"

        ) from e


    return (

        local_index,

        Record(

            b,
            g,
            w,
            N,
            i0,

            p,

            m,
            m2,
            v,

            ok,
            info

        )

    )


def load_or_make_resumable(
    name,
    configs
):

    full_path = (

        EXACT_CACHE

        /

        f"{name}_full.pkl"

    )


    # -------------------------------------------------------------------------
    # Complete new cache
    # -------------------------------------------------------------------------

    if full_path.exists():

        x = safe_pickle_load(
            full_path
        )


        if records_match_configs(
            x,
            configs
        ):

            print(
                f"{name}: full Drive cache loaded "
                f"({len(x):,})"
            )

            return x


        else:

            print(
                f"{name}: existing full cache is incompatible; "
                "ignoring it."
            )


    chunk_dir = (

        EXACT_CACHE

        /

        f"{name}_chunks"

    )


    chunk_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    ans = []


    print(
        f"\n{name}: "
        f"{len(configs):,} exact targets"
    )


    for start in range(
        0,
        len(configs),
        EXACT_CHUNK
    ):

        end = min(
            start+EXACT_CHUNK,
            len(configs)
        )


        chunk_configs = (
            configs[
                start:end
            ]
        )


        chunk_file = (

            chunk_dir

            /

            f"chunk_{start:06d}_{end:06d}.pkl"

        )


        part = None


        # ---------------------------------------------------------------------
        # Reuse completed chunk
        # ---------------------------------------------------------------------

        if chunk_file.exists():

            part = safe_pickle_load(
                chunk_file
            )


            if (
                part is not None
                and
                records_match_configs(
                    part,
                    chunk_configs
                )
            ):

                print(
                    f"[{name:20s}] "
                    f"{start:6d}:{end:6d} | "
                    "Drive cache"
                )


            else:

                part = None


        # ---------------------------------------------------------------------
        # Compute missing chunk
        # ---------------------------------------------------------------------

        if part is None:

            jobs = list(
                enumerate(
                    chunk_configs
                )
            )


            # Large N first inside each chunk.
            jobs.sort(
                key=lambda z:
                z[1][3],
                reverse=True
            )


            t0 = (
                time.perf_counter()
            )


            result = Parallel(

                n_jobs=N_EXACT,

                backend="threading"

            )(

                delayed(
                    _exact_record
                )(
                    j,
                    x
                )

                for j,x in jobs

            )


            result.sort(
                key=lambda z:
                z[0]
            )


            part = [

                r

                for _,r
                in result

            ]


            atomic_pickle(
                part,
                chunk_file
            )


            unresolved = sum(
                not r.tau_valid
                for r in part
            )


            print(
                f"[{name:20s}] "
                f"{start:6d}:{end:6d} | "
                f"tau unresolved={unresolved:3d} | "
                f"{time.perf_counter()-t0:.1f}s | "
                "saved"
            )


        ans.extend(
            part
        )


    if not records_match_configs(
        ans,
        configs
    ):

        raise RuntimeError(
            f"{name}: assembled exact dataset "
            "does not match requested design."
        )


    atomic_pickle(
        ans,
        full_path
    )


    print(
        f"{name}: COMPLETE and permanently saved."
    )


    return ans


# =====================================================================================
# 16. TRY TO REUSE A VERIFIED PREVIOUS EXACT DATASET
# =====================================================================================

def find_verified_previous_dataset(
    patterns,
    configs,
    description
):

    if not OLD_EXACT_CACHE.exists():

        return None


    candidates = []


    for pattern in patterns:

        candidates.extend(
            OLD_EXACT_CACHE.glob(
                pattern
            )
        )


    # Unique paths only
    candidates = list(
        dict.fromkeys(
            candidates
        )
    )


    for path in candidates:

        if not path.is_file():

            continue


        print(
            f"Checking previous {description}:",
            path.name
        )


        x = safe_pickle_load(
            path
        )


        if records_match_configs(
            x,
            configs
        ):

            print(
                f"VERIFIED previous {description}."
            )

            return x


    return None


# =====================================================================================
# 17. BUILD TRAINING DESIGNS
#
# CRITICAL:
# Do NOT use design(100000, seed+1), because that would change the original
# 50,000-point Latin hypercube.
#
# Instead:
#
#   original 50k + new independent 50k
# =====================================================================================

train_design_base = design(

    N_BASE,

    cfg.train_N,

    cfg.seed+1

)


train_design_extra = design(

    N_EXTRA,

    cfg.train_N,

    cfg.seed+1001

)


valid_design = design(

    cfg.n_val,

    cfg.train_N,

    cfg.seed+2

)


test_seen_design = design(

    cfg.n_test_seen,

    EXACT_TEST_N,

    cfg.seed+3

)


test_interp_design = design(

    cfg.n_test_interp,

    INTERP_TEST_N,

    cfg.seed+4

)


# =====================================================================================
# 18. ORIGINAL 50k EXACT TARGETS
# =====================================================================================

previous_train = find_verified_previous_dataset(

    (
        "train_*.pkl",
        "train_base*.pkl"
    ),

    train_design_base,

    "50k training dataset"

)


OLD_BASE_VERIFIED = (
    previous_train is not None
)


if OLD_BASE_VERIFIED:

    train_base = (
        previous_train
    )


    current_base_path = (

        EXACT_CACHE

        /

        "train_base_R50000_full.pkl"

    )


    if not current_base_path.exists():

        atomic_pickle(
            train_base,
            current_base_path
        )


    print(
        "\nOriginal 50,000 exact training targets "
        "reused from previous experiment."
    )


else:

    print(
        "\nNo verified previous 50k training cache found."
    )

    print(
        "The deterministic original 50k design will be generated."
    )


    train_base = load_or_make_resumable(

        "train_base_R50000",

        train_design_base

    )


# =====================================================================================
# 19. ADDITIONAL 50k EXACT TARGETS
# =====================================================================================

train_extra = load_or_make_resumable(

    "train_extra_R50000",

    train_design_extra

)


# =====================================================================================
# 20. ASSEMBLE 100k TRAINING SET
# =====================================================================================

train = (
    train_base
    +
    train_extra
)


assert len(train)==100000


print(
    "\nTraining data assembled:"
)

print(
    "Original:",
    f"{len(train_base):,}"
)

print(
    "Additional:",
    f"{len(train_extra):,}"
)

print(
    "Total:",
    f"{len(train):,}"
)


# =====================================================================================
# 21. VALIDATION / TEST — REUSE OLD COMPATIBLE DATA WHEN POSSIBLE
# =====================================================================================

previous_valid = find_verified_previous_dataset(

    (
        "validation_*.pkl",
        "VALIDATION*.pkl"
    ),

    valid_design,

    "validation dataset"

)


if previous_valid is not None:

    valid = (
        previous_valid
    )


    atomic_pickle(

        valid,

        EXACT_CACHE
        /
        "validation_full.pkl"

    )


else:

    valid = load_or_make_resumable(

        "validation",

        valid_design

    )


previous_test_seen = find_verified_previous_dataset(

    (
        "test_exact_*.pkl",
        "TEST_EXACT*.pkl"
    ),

    test_seen_design,

    "exact-grid test dataset"

)


if previous_test_seen is not None:

    test_seen = (
        previous_test_seen
    )


    atomic_pickle(

        test_seen,

        EXACT_CACHE
        /
        "test_exact_full.pkl"

    )


else:

    test_seen = load_or_make_resumable(

        "test_exact",

        test_seen_design

    )


previous_test_interp = find_verified_previous_dataset(

    (
        "test_interp_*.pkl",
        "TEST_INTERP*.pkl"
    ),

    test_interp_design,

    "interpolation test dataset"

)


if previous_test_interp is not None:

    test_interp = (
        previous_test_interp
    )


    atomic_pickle(

        test_interp,

        EXACT_CACHE
        /
        "test_interp_full.pkl"

    )


else:

    test_interp = load_or_make_resumable(

        "test_interp",

        test_interp_design

    )


# =====================================================================================
# 22. AUDIT
# =====================================================================================

def audit(
    records,
    name
):

    bad = [

        r

        for r in records

        if (

            not np.all(
                np.isfinite(
                    r.p
                )
            )

            or

            abs(
                r.p.sum()-1.
            )>1e-6

        )

    ]


    if bad:

        raise RuntimeError(
            f"Invalid probability targets in {name}."
        )


    print(
        "\n"+name
    )


    for N in sorted(
        {
            r.N
            for r in records
        }
    ):

        x = [

            r

            for r in records

            if r.N==N

        ]


        print(
            f"N={N:3d} | "
            f"n={len(x):6d} | "
            f"i0=1={sum(r.i0==1 for r in x):6d} | "
            f"valid tau="
            f"{sum(r.tau_valid for r in x):6d}/{len(x):6d}"
        )


print(
    "\n"
    +
    "="*115
)

print(
    "EXACT-TARGET AUDIT"
)

print(
    "="*115
)


audit(
    train_base,
    "TRAIN — ORIGINAL 50k"
)

audit(
    train_extra,
    "TRAIN — ADDITIONAL 50k"
)

audit(
    valid,
    "VALIDATION"
)

audit(
    test_seen,
    "TEST — EXACT N"
)

audit(
    test_interp,
    "TEST — INTERPOLATED N"
)


# =====================================================================================
# 23. NETWORKS
# =====================================================================================

def mlp(
    din,
    dout
):

    L=[]

    d=din


    for _ in range(
        cfg.depth
    ):

        L += [

            nn.Linear(
                d,
                cfg.width
            ),

            nn.SiLU()

        ]


        d=(
            cfg.width
        )


    L.append(

        nn.Linear(
            d,
            dout
        )

    )


    return nn.Sequential(
        *L
    )


class HazardNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.net=mlp(
            6,
            1
        )


    def forward(
        self,
        x
    ):

        return torch.sigmoid(

            self.net(
                x
            ).squeeze(-1)

        )


class TauNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.net=mlp(
            5,
            2
        )


    def forward(
        self,
        x
    ):

        return torch.nn.functional.softplus(

            self.net(
                x
            )

        )


# =====================================================================================
# 24. NETWORK INPUTS
# =====================================================================================

def xtau(r):

    return torch.tensor(

        [

            r.beta,
            r.gamma,
            r.omega,

            r.N/N_scale,

            r.i0/r.N

        ],

        dtype=torch.float32,

        device=device

    )


def xhaz(r):

    c = torch.arange(

        r.N+1,

        dtype=torch.float32,

        device=device

    )


    n=(
        r.N+1
    )


    return torch.column_stack(

        [

            torch.full(
                (n,),
                r.beta,
                device=device
            ),

            torch.full(
                (n,),
                r.gamma,
                device=device
            ),

            torch.full(
                (n,),
                r.omega,
                device=device
            ),

            torch.full(
                (n,),
                r.N/N_scale,
                device=device
            ),

            torch.full(
                (n,),
                r.i0/r.N,
                device=device
            ),

            c/r.N

        ]

    )


# =====================================================================================
# 25. HAZARD RECONSTRUCTION
# =====================================================================================

def reconstruct(h):

    before = torch.cat(

        [

            torch.ones(
                1,
                device=h.device
            ),

            torch.cumprod(
                1-h[:-1],
                0
            )

        ]

    )


    return torch.cat(

        [

            before*h,

            torch.prod(
                1-h
            ).reshape(1)

        ]

    )


def tail_torch(p):

    return torch.flip(

        torch.cumsum(

            torch.flip(
                p[1:],
                dims=[0]
            ),

            dim=0

        ),

        dims=[0]

    )


def phat_tensor(
    model,
    r
):

    return reconstruct(

        model(
            xhaz(r)
        )

    )


def tau_target(r):

    if not r.tau_valid:

        raise RuntimeError(
            "tau_target called for unresolved tau target."
        )


    raw = np.asarray(

        [
            r.mean_tau,
            r.var_tau
        ],

        dtype=np.float64

    )


    if (
        not np.all(
            np.isfinite(
                raw
            )
        )
        or
        raw[0]<=0
        or
        raw[1]<0
    ):

        raise RuntimeError(
            f"Invalid tau target: {raw}"
        )


    return torch.tensor(

        np.log1p(
            raw
        ),

        dtype=torch.float32,

        device=device

    )


# =====================================================================================
# 26. LOSS
# =====================================================================================

def batch_loss(
    records,
    ix,
    hnet,
    tnet
):

    LP=[]
    LRHO=[]
    LT=[]


    for j in ix:

        r=records[
            int(j)
        ]


        p=torch.tensor(

            r.p,

            dtype=torch.float32,

            device=device

        )


        ph=phat_tensor(
            hnet,
            r
        )


        # PMF
        LP.append(

            torch.sum(
                (ph-p)**2
            )

        )


        # Tail risk
        rho=tail_torch(
            p
        )


        rhoh=tail_torch(
            ph
        )


        LRHO.append(

            torch.mean(
                (rhoh-rho)**2
            )

        )


        # Extinction time
        if r.tau_valid:

            z=tau_target(
                r
            )


            zh=tnet(

                xtau(
                    r
                ).unsqueeze(0)

            ).squeeze(0)


            LT.append(

                torch.sum(

                    (zh-z)**2

                    /

                    (
                        1
                        +
                        z*z
                    )

                )

            )


    lp=torch.stack(
        LP
    ).mean()


    lrho=torch.stack(
        LRHO
    ).mean()


    lt=(

        torch.stack(
            LT
        ).mean()

        if LT

        else

        torch.zeros(
            (),
            device=device
        )

    )


    total=(

        lp

        +

        cfg.lambda_rho
        *
        lrho

        +

        cfg.lambda_tau
        *
        lt

    )


    return (
        total,
        lp,
        lrho,
        lt
    )


# =====================================================================================
# 27. VALIDATION LOSS
# =====================================================================================

@torch.no_grad()
def validation_loss(
    hnet,
    tnet
):

    hnet.eval()
    tnet.eval()


    L,_,_,_=batch_loss(

        valid,

        np.arange(
            len(valid)
        ),

        hnet,
        tnet

    )


    return L.item()


# =====================================================================================
# 28. RESUMABLE TRAINING
#
# R=100,000 / batch 64 gives about 1,563 mini-batches per epoch.
#
# We therefore checkpoint every 50 mini-batches.
# =====================================================================================

BATCH_CHECKPOINT_EVERY=50


def cpu_state_dict(
    state
):

    return {

        k:
        v.detach().cpu()

        for k,v
        in state.items()

    }


def train_emulator_resumable(
    records,
    seed,
    R,
    verbose=True
):

    final_file=(

        MODEL_CACHE

        /

        f"R{R}_FINAL.pt"

    )


    checkpoint_file=(

        CHECKPOINT_CACHE

        /

        f"R{R}_CHECKPOINT.pt"

    )


    # =========================================================================
    # FINISHED MODEL
    # =========================================================================

    if final_file.exists():

        ck=torch.load(

            final_file,

            map_location=device,

            weights_only=False

        )


        hnet=HazardNet().to(
            device
        )


        tnet=TauNet().to(
            device
        )


        hnet.load_state_dict(
            ck["hazard"]
        )


        tnet.load_state_dict(
            ck["tau"]
        )


        print(
            f"R={R}: FINAL model loaded | "
            f"best epoch={ck['best_epoch']} | "
            f"stop epoch={ck['stopped_epoch']}"
        )


        return {

            "hazard":
                hnet,

            "tau":
                tnet,

            "history":
                ck["history"],

            "training_time":
                float(
                    ck["training_time"]
                ),

            "best_validation_loss":
                float(
                    ck["best_validation_loss"]
                ),

            "best_epoch":
                int(
                    ck["best_epoch"]
                ),

            "stopped_epoch":
                int(
                    ck["stopped_epoch"]
                ),

            "n_tau_train":
                int(
                    ck["n_tau_train"]
                )

        }


    # =========================================================================
    # INITIAL MODEL
    # =========================================================================

    seed_all(
        seed
    )


    hnet=HazardNet().to(
        device
    )


    tnet=TauNet().to(
        device
    )


    pars=(

        list(
            hnet.parameters()
        )

        +

        list(
            tnet.parameters()
        )

    )


    opt=torch.optim.AdamW(

        pars,

        lr=cfg.lr,

        weight_decay=cfg.weight_decay

    )


    scheduler=(

        torch.optim.lr_scheduler.ReduceLROnPlateau(

            opt,

            mode="min",

            factor=.5,

            patience=15

        )

    )


    rng=np.random.default_rng(
        seed+7011
    )


    best=np.inf

    best_epoch=None

    bh=None
    bt=None

    wait=0


    history={

        "joint":[],
        "pmf":[],
        "tail":[],
        "tau":[],
        "validation":[]

    }


    start_epoch=1

    resume_perm=None

    resume_batch_start=0


    J=[]
    P=[]
    RHO=[]
    TT=[]


    previous_elapsed=0.


    # =========================================================================
    # RESTORE CHECKPOINT
    # =========================================================================

    if checkpoint_file.exists():

        ck=torch.load(

            checkpoint_file,

            map_location=device,

            weights_only=False

        )


        hnet.load_state_dict(
            ck["hazard_current"]
        )


        tnet.load_state_dict(
            ck["tau_current"]
        )


        opt.load_state_dict(
            ck["optimizer"]
        )


        scheduler.load_state_dict(
            ck["scheduler"]
        )


        best=(
            ck["best"]
        )


        best_epoch=(
            ck["best_epoch"]
        )


        bh=(
            ck["best_hazard"]
        )


        bt=(
            ck["best_tau"]
        )


        wait=(
            ck["wait"]
        )


        history=(
            ck["history"]
        )


        start_epoch=int(
            ck["epoch"]
        )


        resume_perm=(
            ck["perm"]
        )


        resume_batch_start=int(
            ck["batch_start"]
        )


        J=list(
            ck.get(
                "J",
                []
            )
        )


        P=list(
            ck.get(
                "P",
                []
            )
        )


        RHO=list(
            ck.get(
                "RHO",
                []
            )
        )


        TT=list(
            ck.get(
                "T",
                []
            )
        )


        previous_elapsed=float(
            ck.get(
                "elapsed_sec",
                0.
            )
        )


        rng.bit_generator.state=(
            ck["rng_state"]
        )


        if "torch_rng_state" in ck:

            torch.set_rng_state(
                ck["torch_rng_state"].cpu()
            )


        if (
            device.type=="cuda"
            and
            ck.get(
                "cuda_rng_states"
            ) is not None
        ):

            torch.cuda.set_rng_state_all(
                ck[
                    "cuda_rng_states"
                ]
            )


        print(
            "\n"
            f"R={R}: RESUMING TRAINING"
        )


        print(
            f"epoch={start_epoch} | "
            f"next sample={resume_batch_start:,} | "
            f"best epoch={best_epoch} | "
            f"best val={best:.4e} | "
            f"wait={wait}"
        )


    session_start=(
        time.perf_counter()
    )


    stopped_epoch=None


    # =========================================================================
    # TRAINING
    # =========================================================================

    for epoch in range(
        start_epoch,
        cfg.epochs+1
    ):

        hnet.train()
        tnet.train()


        # ---------------------------------------------------------------------
        # Continue interrupted epoch
        # ---------------------------------------------------------------------

        if (
            epoch==start_epoch
            and
            resume_perm is not None
        ):

            perm=np.asarray(
                resume_perm,
                dtype=int
            )


            batch_start=(
                resume_batch_start
            )


        else:

            perm=rng.permutation(
                len(records)
            )


            batch_start=0


            J=[]
            P=[]
            RHO=[]
            TT=[]


        batch_number=(
            batch_start
            //
            cfg.batch_size
        )


        for s in range(

            batch_start,

            len(perm),

            cfg.batch_size

        ):

            ix=perm[
                s:
                s+cfg.batch_size
            ]


            opt.zero_grad(
                set_to_none=True
            )


            (
                L,
                Lp,
                Lrho,
                Lt

            )=batch_loss(

                records,
                ix,
                hnet,
                tnet

            )


            if not torch.isfinite(
                L
            ):

                raise RuntimeError(

                    f"Non-finite loss | "
                    f"R={R}, "
                    f"epoch={epoch}, "
                    f"sample={s}"

                )


            L.backward()


            torch.nn.utils.clip_grad_norm_(

                pars,

                cfg.grad_clip

            )


            opt.step()


            J.append(
                L.detach().item()
            )


            P.append(
                Lp.detach().item()
            )


            RHO.append(
                Lrho.detach().item()
            )


            TT.append(
                Lt.detach().item()
            )


            batch_number+=1


            next_start=min(

                s
                +
                cfg.batch_size,

                len(perm)

            )


            # =================================================================
            # MID-EPOCH PERMANENT CHECKPOINT
            # =================================================================

            if (
                batch_number
                %
                BATCH_CHECKPOINT_EVERY
                ==
                0

                and

                next_start
                <
                len(perm)
            ):

                elapsed=(

                    previous_elapsed

                    +

                    (
                        time.perf_counter()
                        -
                        session_start
                    )

                )


                atomic_torch_save(

                    {

                        "epoch":
                            epoch,

                        "perm":
                            perm,

                        "batch_start":
                            next_start,

                        "J":
                            J,

                        "P":
                            P,

                        "RHO":
                            RHO,

                        "T":
                            TT,

                        "hazard_current":
                            cpu_state_dict(
                                hnet.state_dict()
                            ),

                        "tau_current":
                            cpu_state_dict(
                                tnet.state_dict()
                            ),

                        "optimizer":
                            opt.state_dict(),

                        "scheduler":
                            scheduler.state_dict(),

                        "best":
                            best,

                        "best_epoch":
                            best_epoch,

                        "best_hazard":
                            bh,

                        "best_tau":
                            bt,

                        "wait":
                            wait,

                        "history":
                            history,

                        "rng_state":
                            rng.bit_generator.state,

                        "torch_rng_state":
                            torch.get_rng_state(),

                        "cuda_rng_states":
                            (
                                torch.cuda.get_rng_state_all()

                                if device.type=="cuda"

                                else None
                            ),

                        "elapsed_sec":
                            elapsed

                    },

                    checkpoint_file

                )


                print(
                    f"R={R:6d} | "
                    f"epoch={epoch:3d} | "
                    f"batch={batch_number:4d} | "
                    f"sample={next_start:7d}/{len(perm):7d} | "
                    "CHECKPOINT"
                )


        # =====================================================================
        # EPOCH COMPLETE
        # =====================================================================

        j=float(
            np.mean(J)
        )


        p=float(
            np.mean(P)
        )


        rho=float(
            np.mean(RHO)
        )


        tau=float(
            np.mean(TT)
        )


        v=validation_loss(
            hnet,
            tnet
        )


        scheduler.step(
            v
        )


        history[
            "joint"
        ].append(
            j
        )


        history[
            "pmf"
        ].append(
            p
        )


        history[
            "tail"
        ].append(
            rho
        )


        history[
            "tau"
        ].append(
            tau
        )


        history[
            "validation"
        ].append(
            v
        )


        # ---------------------------------------------------------------------
        # Best validation model
        # ---------------------------------------------------------------------

        if (
            bh is None
            or
            v<best-cfg.min_delta
        ):

            best=v

            best_epoch=epoch

            wait=0


            bh=cpu_state_dict(
                hnet.state_dict()
            )


            bt=cpu_state_dict(
                tnet.state_dict()
            )


        else:

            wait+=1


        if verbose:

            print(
                f"R={R:6d} | "
                f"epoch={epoch:4d} | "
                f"joint={j:.3e} | "
                f"pmf={p:.3e} | "
                f"tail={rho:.3e} | "
                f"tau={tau:.3e} | "
                f"val={v:.3e} | "
                f"best_epoch={best_epoch} | "
                f"wait={wait}"
            )


        # =====================================================================
        # END-OF-EPOCH CHECKPOINT
        # =====================================================================

        elapsed=(

            previous_elapsed

            +

            (
                time.perf_counter()
                -
                session_start
            )

        )


        atomic_torch_save(

            {

                # Start next epoch after reconnect.
                "epoch":
                    epoch+1,

                "perm":
                    None,

                "batch_start":
                    0,

                "J":
                    [],

                "P":
                    [],

                "RHO":
                    [],

                "T":
                    [],

                "hazard_current":
                    cpu_state_dict(
                        hnet.state_dict()
                    ),

                "tau_current":
                    cpu_state_dict(
                        tnet.state_dict()
                    ),

                "optimizer":
                    opt.state_dict(),

                "scheduler":
                    scheduler.state_dict(),

                "best":
                    best,

                "best_epoch":
                    best_epoch,

                "best_hazard":
                    bh,

                "best_tau":
                    bt,

                "wait":
                    wait,

                "history":
                    history,

                "rng_state":
                    rng.bit_generator.state,

                "torch_rng_state":
                    torch.get_rng_state(),

                "cuda_rng_states":
                    (
                        torch.cuda.get_rng_state_all()

                        if device.type=="cuda"

                        else None
                    ),

                "elapsed_sec":
                    elapsed

            },

            checkpoint_file

        )


        if wait>=cfg.patience:

            stopped_epoch=epoch


            print(
                f"Early stopping R={R}: "
                f"stop epoch={stopped_epoch}, "
                f"best epoch={best_epoch}, "
                f"best val={best:.4e}"
            )


            break


        resume_perm=None

        resume_batch_start=0

        J=[]
        P=[]
        RHO=[]
        TT=[]


    if stopped_epoch is None:

        stopped_epoch=min(
            cfg.epochs,
            epoch
        )


    if (
        bh is None
        or
        bt is None
    ):

        raise RuntimeError(
            f"R={R}: no valid best model state."
        )


    hnet.load_state_dict(
        bh
    )


    tnet.load_state_dict(
        bt
    )


    training_time=(

        previous_elapsed

        +

        (
            time.perf_counter()
            -
            session_start
        )

    )


    final_payload={

        "hazard":
            bh,

        "tau":
            bt,

        "history":
            history,

        "training_time":
            training_time,

        "best_validation_loss":
            best,

        "best_epoch":
            best_epoch,

        "stopped_epoch":
            stopped_epoch,

        "n_tau_train":
            sum(
                r.tau_valid
                for r in records
            )

    }


    atomic_torch_save(
        final_payload,
        final_file
    )


    if checkpoint_file.exists():

        checkpoint_file.unlink()


    print(
        f"R={R}: FINAL model permanently saved."
    )


    return {

        "hazard":
            hnet,

        "tau":
            tnet,

        "history":
            history,

        "training_time":
            training_time,

        "best_validation_loss":
            best,

        "best_epoch":
            best_epoch,

        "stopped_epoch":
            stopped_epoch,

        "n_tau_train":
            final_payload[
                "n_tau_train"
            ]

    }


# =====================================================================================
# 29. EVALUATION
# =====================================================================================

def tail_np(p):

    return np.flip(

        np.cumsum(

            np.flip(
                p[1:]
            )

        )

    )


@torch.no_grad()
def predict(
    r,
    hnet,
    tnet
):

    hnet.eval()
    tnet.eval()


    p=phat_tensor(
        hnet,
        r
    ).cpu().numpy()


    z=(

        tnet(

            xtau(
                r
            ).unsqueeze(0)

        )

        .squeeze(0)

        .cpu()

        .numpy()

        .astype(
            np.float64
        )

    )


    moments=np.expm1(

        np.clip(
            z,
            0.,
            700.
        )

    )


    return (

        p,

        float(
            moments[0]
        ),

        float(
            moments[1]
        )

    )


def evaluate(
    records,
    hnet,
    tnet,
    split
):

    ans=[]


    for j,r in enumerate(
        records
    ):

        p,m,v=predict(
            r,
            hnet,
            tnet
        )


        rho=tail_np(
            r.p
        )


        rhoh=tail_np(
            p
        )


        pos=(
            r.p>0
        )


        psafe=np.clip(
            p,
            cfg.kl_eps,
            1.
        )


        R0=(
            r.beta/r.gamma
        )


        row={

            "index":
                j,

            "split":
                split,

            "N":
                r.N,

            "i0":
                r.i0,

            "beta":
                r.beta,

            "gamma":
                r.gamma,

            "omega":
                r.omega,

            "R0":
                R0,

            "tau_valid":
                r.tau_valid,

            "E2":
                float(
                    np.linalg.norm(
                        p-r.p
                    )
                ),

            "E_rho":
                float(
                    np.max(
                        np.abs(
                            rhoh-rho
                        )
                    )
                ),

            "E_overflow":
                float(
                    abs(
                        p[-1]
                        -
                        r.p[-1]
                    )
                ),

            "KL":
                float(

                    np.sum(

                        r.p[pos]

                        *

                        np.log(
                            r.p[pos]
                            /
                            psafe[pos]
                        )

                    )

                ),

            "exact_p":
                r.p,

            "pred_p":
                p,

            "exact_tail":
                rho,

            "pred_tail":
                rhoh

        }


        if r.tau_valid:

            row[
                "mean_tau_relative_error"
            ]=float(

                abs(
                    m-r.mean_tau
                )

                /

                r.mean_tau

            )


            row[
                "var_tau_relative_error"
            ]=float(

                abs(
                    v-r.var_tau
                )

                /

                max(
                    r.var_tau,
                    1e-300
                )

            )


        else:

            row[
                "mean_tau_relative_error"
            ]=np.nan


            row[
                "var_tau_relative_error"
            ]=np.nan


        ans.append(
            row
        )


    return ans


METRICS=[

    "E2",
    "E_rho",
    "E_overflow",
    "KL",
    "mean_tau_relative_error",
    "var_tau_relative_error"

]


def finite_values(
    results,
    key
):

    x=np.asarray(

        [
            r[key]
            for r in results
        ],

        dtype=float

    )


    return x[
        np.isfinite(
            x
        )
    ]


def median_metric(
    results,
    key
):

    x=finite_values(
        results,
        key
    )


    return (

        float(
            np.median(
                x
            )
        )

        if len(x)

        else np.nan

    )


# =====================================================================================
# 30. BALANCED NESTED ORDER
# =====================================================================================

def balanced_order(
    records,
    seed
):

    rng=np.random.default_rng(
        seed
    )


    groups={}


    for N in cfg.train_N:

        for flag in (
            0,
            1
        ):

            x=np.asarray(

                [

                    j

                    for j,r
                    in enumerate(
                        records
                    )

                    if
                    r.N==N
                    and
                    int(r.i0==1)==flag

                ],

                dtype=int

            )


            rng.shuffle(
                x
            )


            groups[
                (N,flag)
            ]=list(
                x
            )


    ptr={
        k:0
        for k in groups
    }


    usedN={
        N:0
        for N in cfg.train_N
    }


    used1=0

    order=[]


    for position in range(
        len(records)
    ):

        target1=(

            cfg.i0_one_fraction

            *

            (
                position+1
            )

        )


        preferred_flag=(

            1

            if used1<target1

            else 0

        )


        selected=None


        for flag in (
            preferred_flag,
            1-preferred_flag
        ):

            candidates=[

                N

                for N in cfg.train_N

                if

                ptr[
                    (N,flag)
                ]

                <

                len(
                    groups[
                        (N,flag)
                    ]
                )

            ]


            if candidates:

                min_used=min(
                    usedN[N]
                    for N in candidates
                )


                candidates=[

                    N

                    for N in candidates

                    if
                    usedN[N]==min_used

                ]


                N=int(
                    rng.choice(
                        candidates
                    )
                )


                selected=(
                    N,
                    flag
                )


                break


        if selected is None:

            raise RuntimeError(
                "Could not construct balanced nested order."
            )


        N,flag=selected


        j=groups[
            (N,flag)
        ][
            ptr[
                (N,flag)
            ]
        ]


        ptr[
            (N,flag)
        ]+=1


        usedN[
            N
        ]+=1


        used1+=flag


        order.append(
            j
        )


    return np.asarray(
        order,
        dtype=int
    )


# =====================================================================================
# 31. PRESERVE ORIGINAL 50k ORDER + APPEND NEW 50k ORDER
# =====================================================================================

order_base=balanced_order(

    train_base,

    cfg.seed+20000

)


order_extra_local=balanced_order(

    train_extra,

    cfg.seed+21000

)


# train_extra starts at global index 50,000.
order_extra=(

    N_BASE

    +

    order_extra_local

)


order=np.concatenate(

    [
        order_base,
        order_extra
    ]

)


assert len(order)==100000

assert len(
    np.unique(
        order
    )
)==100000


# Explicit preservation check
assert np.array_equal(

    order[
        :N_BASE
    ],

    order_base

)


print(
    "\nNested order successfully constructed."
)

print(
    "First 50,000 positions preserve the original experiment: YES"
)

print(
    "Additional 50,000 positions appended: YES"
)


# =====================================================================================
# 32. SHOWCASE CONFIGURATIONS
# =====================================================================================

def entropy(p):

    z=p[
        p>0
    ]


    return float(

        -np.sum(
            z*np.log(z)
        )

    )


def bimodality(p):

    z=(
        p[:-1]
    )


    if (
        len(z)<3
        or
        z.max()<=0
    ):

        return 0.


    peaks=[

        j

        for j in range(
            len(z)
        )

        if

        z[j]
        >=
        (
            z[j-1]
            if j
            else -np.inf
        )

        and

        z[j]
        >=
        (
            z[j+1]
            if j<len(z)-1
            else -np.inf
        )

        and

        z[j]>=.03*z.max()

    ]


    if len(peaks)<2:

        return 0.


    a,b=sorted(

        peaks,

        key=lambda j:
        z[j],

        reverse=True

    )[:2]


    return float(

        min(
            z[a],
            z[b]
        )

        /

        max(
            z[a],
            z[b]
        )

        *

        abs(a-b)

        /

        max(
            len(z)-1,
            1
        )

    )


SHOWCASE_N=max(
    EXACT_TEST_N
)


exact_large_i1=[

    r

    for r in test_seen

    if
    r.N==SHOWCASE_N
    and
    r.i0==1

]


if exact_large_i1:

    case1=max(

        exact_large_i1,

        key=lambda r:
        entropy(
            r.p
        )

    )


else:

    exact_large=[

        r

        for r in test_seen

        if r.N==SHOWCASE_N

    ]


    if not exact_large:

        raise RuntimeError(
            f"No exact-test records at N={SHOWCASE_N}."
        )


    case1=max(

        exact_large,

        key=lambda r:
        entropy(
            r.p
        )

    )


if not test_interp:

    raise RuntimeError(
        "Interpolation test set is empty."
    )


case2=max(

    test_interp,

    key=lambda r:
    bimodality(
        r.p
    )

)


SHOWCASES=[

    (

        rf"Exact-grid $N={SHOWCASE_N},\ i_0={case1.i0}$",

        case1

    ),

    (

        "Held-out interpolation",

        case2

    )

]


print(
    "\nFixed showcase configurations"
)

print(
    "-"*75
)

print(
    "Showcase 1:",
    f"N={case1.N}, "
    f"i0={case1.i0}, "
    f"R0={case1.beta/case1.gamma:.3f}"
)

print(
    "Showcase 2:",
    f"N={case2.N}, "
    f"i0={case2.i0}, "
    f"R0={case2.beta/case2.gamma:.3f}"
)


# =====================================================================================
# 33. CURRENT EXPERIMENT PROGRESS
# =====================================================================================

PROGRESS_FILE=(

    ROOT

    /

    "learning_curve_progress.pkl"

)


progress=safe_pickle_load(

    PROGRESS_FILE,

    default=None

)


if progress is None:

    progress={
        "results":{}
    }


if "results" not in progress:

    progress[
        "results"
    ]={}


# =====================================================================================
# 34. IMPORT PREVIOUS R<=50k RESULTS / MODELS
#
# Only done if the previous 50k exact dataset was explicitly verified.
# =====================================================================================

if OLD_BASE_VERIFIED:

    print(
        "\nPrevious 50k design verified."
    )

    print(
        "Attempting to reuse R=20k,30k,50k models/results..."
    )


    # -------------------------------------------------------------------------
    # Models
    # -------------------------------------------------------------------------

    for R in (
        20000,
        30000,
        50000
    ):

        src=(

            OLD_MODEL_CACHE

            /

            f"R{R}_FINAL.pt"

        )


        dst=(

            MODEL_CACHE

            /

            f"R{R}_FINAL.pt"

        )


        if (
            src.exists()
            and
            not dst.exists()
        ):

            shutil.copy2(
                src,
                dst
            )


            print(
                f"Copied previous R={R} model."
            )


    # -------------------------------------------------------------------------
    # Completed result rows / galleries / histories
    # -------------------------------------------------------------------------

    if OLD_PROGRESS_FILE.exists():

        old_progress=safe_pickle_load(
            OLD_PROGRESS_FILE
        )


        if old_progress is not None:

            old_results=old_progress.get(
                "results",
                {}
            )


            for R in (
                20000,
                30000,
                50000
            ):

                old_z=None


                if R in old_results:

                    old_z=old_results[
                        R
                    ]


                elif str(R) in old_results:

                    old_z=old_results[
                        str(R)
                    ]


                if (
                    old_z is not None
                    and
                    R not in progress[
                        "results"
                    ]
                ):

                    progress[
                        "results"
                    ][R]=old_z


                    print(
                        f"Imported previous R={R} result."
                    )


            atomic_pickle(
                progress,
                PROGRESS_FILE
            )


else:

    print(
        "\nPrevious 50k dataset was not explicitly verified."
    )

    print(
        "Previous R<=50k models/results will NOT be imported automatically."
    )


# =====================================================================================
# 35. LEARNING-CURVE EXPERIMENT
# =====================================================================================

for R in R_VALUES:

    # -------------------------------------------------------------------------
    # Already completed
    # -------------------------------------------------------------------------

    if R in progress[
        "results"
    ]:

        print(
            f"\nR={R:,}: "
            "evaluation already complete — skipping."
        )

        continue


    # -------------------------------------------------------------------------
    # Nested subset
    # -------------------------------------------------------------------------

    subset=[

        train[
            int(j)
        ]

        for j in order[
            :R
        ]

    ]


    counts={

        N:
        sum(
            r.N==N
            for r in subset
        )

        for N in cfg.train_N

    }


    n_i1=sum(
        r.i0==1
        for r in subset
    )


    n_tau=sum(
        r.tau_valid
        for r in subset
    )


    print(
        "\n"
        +
        "="*115
    )


    print(
        f"TRAINING R={R:,}"
    )


    print(
        "="*115
    )


    print(
        "N counts:",
        counts
    )


    print(
        f"i0=1: "
        f"{n_i1:,}/{R:,} "
        f"({n_i1/R:.1%})"
    )


    print(
        f"valid tau: "
        f"{n_tau:,}/{R:,}"
    )


    # -------------------------------------------------------------------------
    # Resumable neural training
    # -------------------------------------------------------------------------

    fit=train_emulator_resumable(

        subset,

        seed=(
            cfg.seed
            +
            30000
            +
            R
        ),

        R=R,

        verbose=True

    )


    hnet=fit[
        "hazard"
    ]


    tnet=fit[
        "tau"
    ]


    # -------------------------------------------------------------------------
    # Test evaluation
    # -------------------------------------------------------------------------

    exact=evaluate(

        test_seen,

        hnet,
        tnet,

        "exact"

    )


    interp=evaluate(

        test_interp,

        hnet,
        tnet,

        "interpolation"

    )


    interp_i1=[

        x

        for x in interp

        if x["i0"]==1

    ]


    interp_i_gt1=[

        x

        for x in interp

        if x["i0"]>1

    ]


    R_rows=[]


    for split,res in (

        (
            "exact",
            exact
        ),

        (
            "interpolation",
            interp
        ),

        (
            "interpolation_i0_1",
            interp_i1
        ),

        (
            "interpolation_i0_gt1",
            interp_i_gt1
        )

    ):

        row={

            "R":
                R,

            "split":
                split,

            "training_time":
                fit[
                    "training_time"
                ],

            "best_validation_loss":
                fit[
                    "best_validation_loss"
                ],

            "best_epoch":
                fit[
                    "best_epoch"
                ],

            "stopped_epoch":
                fit[
                    "stopped_epoch"
                ],

            "n_tau_train":
                fit[
                    "n_tau_train"
                ],

            "n_tau_eval":
                sum(
                    x[
                        "tau_valid"
                    ]
                    for x in res
                )

        }


        for key in METRICS:

            row[
                key
            ]=median_metric(
                res,
                key
            )


        R_rows.append(
            row
        )


    # -------------------------------------------------------------------------
    # Gallery
    # -------------------------------------------------------------------------

    R_gallery={}


    if R in GALLERY_R:

        for label,r in SHOWCASES:

            R_gallery[
                label
            ]=evaluate(

                [r],

                hnet,
                tnet,

                "gallery"

            )[0]


    # -------------------------------------------------------------------------
    # PERMANENT R RESULT
    # -------------------------------------------------------------------------

    progress[
        "results"
    ][R]={

        "rows":
            R_rows,

        "gallery":
            R_gallery,

        "history":
            fit[
                "history"
            ],

        "fit_meta":{

            "training_time":
                fit[
                    "training_time"
                ],

            "best_validation_loss":
                fit[
                    "best_validation_loss"
                ],

            "best_epoch":
                fit[
                    "best_epoch"
                ],

            "stopped_epoch":
                fit[
                    "stopped_epoch"
                ]

        }

    }


    atomic_pickle(
        progress,
        PROGRESS_FILE
    )


    print(
        f"\nR={R:,}: evaluation result "
        "permanently saved."
    )


    print(
        f"Interpolation | "
        f"E2={median_metric(interp,'E2'):.4e} | "
        f"E_rho={median_metric(interp,'E_rho'):.4e} | "
        f"E(tau)="
        f"{median_metric(interp,'mean_tau_relative_error'):.4e} | "
        f"Var(tau)="
        f"{median_metric(interp,'var_tau_relative_error'):.4e}"
    )


    print(
        f"best epoch={fit['best_epoch']} | "
        f"stop epoch={fit['stopped_epoch']} | "
        f"best val={fit['best_validation_loss']:.4e}"
    )


    del (
        fit,
        hnet,
        tnet,
        exact,
        interp,
        interp_i1,
        interp_i_gt1
    )


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# =====================================================================================
# 36. RECONSTRUCT GLOBAL RESULTS
# =====================================================================================

rows=[]


gallery={

    label:{}

    for label,_ in SHOWCASES

}


for R in R_VALUES:

    if R not in progress[
        "results"
    ]:

        continue


    z=progress[
        "results"
    ][R]


    rows.extend(
        z[
            "rows"
        ]
    )


    for label,value in z.get(
        "gallery",
        {}
    ).items():

        gallery[
            label
        ][R]=value


# =====================================================================================
# 37. SAVE SCIENTIFIC SUMMARY TABLE
# =====================================================================================

summary_df=pd.DataFrame(
    rows
)


summary_df.to_csv(

    OUT
    /
    "section_5_1B_R100000_summary.csv",

    index=False

)


(
    OUT
    /
    "section_5_1B_R100000_summary.tex"
).write_text(

    summary_df.to_latex(
        index=False,
        float_format="%.5g"
    )

)


# =====================================================================================
# 38. CURVE HELPER
# =====================================================================================

def curve(
    split,
    key
):

    x=[]
    y=[]


    for R in R_VALUES:

        matches=[

            z[
                key
            ]

            for z in rows

            if
            z["R"]==R
            and
            z["split"]==split

        ]


        if matches:

            x.append(
                R
            )

            y.append(
                matches[0]
            )


    return (

        np.asarray(
            x,
            dtype=float
        ),

        np.asarray(
            y,
            dtype=float
        )

    )


# =====================================================================================
# 39. INTERPOLATION SUMMARY
# =====================================================================================

print(
    "\n"
    +
    "="*135
)

print(
    "HELD-OUT POPULATION-SIZE INTERPOLATION SUMMARY"
)

print(
    "="*135
)


print(
    f"{'R':>9s} | "
    f"{'E2':>11s} | "
    f"{'E_rho':>11s} | "
    f"{'E_over':>11s} | "
    f"{'KL':>11s} | "
    f"{'Rel E(tau)':>12s} | "
    f"{'Rel Var(tau)':>14s} | "
    f"{'Best ep.':>8s} | "
    f"{'Stop ep.':>8s}"
)


print(
    "-"*145
)


for R in R_VALUES:

    found=[

        z

        for z in rows

        if
        z["R"]==R
        and
        z["split"]=="interpolation"

    ]


    if not found:

        continue


    row=found[0]


    print(
        f"{R:9d} | "
        f"{row['E2']:11.4e} | "
        f"{row['E_rho']:11.4e} | "
        f"{row['E_overflow']:11.4e} | "
        f"{row['KL']:11.4e} | "
        f"{row['mean_tau_relative_error']:12.4e} | "
        f"{row['var_tau_relative_error']:14.4e} | "
        f"{row['best_epoch']:8d} | "
        f"{row['stopped_epoch']:8d}"
    )


# =====================================================================================
# 40. FIGURE 1 — EXACT GRID VS INTERPOLATION
# =====================================================================================

plt.rcParams.update({

    "font.size":
        10.5,

    "axes.spines.top":
        False,

    "axes.spines.right":
        False

})


fig,axs=plt.subplots(

    2,
    2,

    figsize=(
        13,
        9
    )

)


spec=[

    (
        "E2",
        r"Median $E_2$",
        "(A) Distributional error"
    ),

    (
        "E_rho",
        r"Median $E_\rho$",
        "(B) Tail-risk error"
    ),

    (
        "mean_tau_relative_error",
        "Median relative error",
        r"(C) $E(\tau)$"
    ),

    (
        "var_tau_relative_error",
        "Median relative error",
        r"(D) $\mathrm{Var}(\tau)$"
    )

]


for ax,(
    key,
    ylabel,
    title

) in zip(
    axs.flat,
    spec
):

    for (
        split,
        label,
        color,
        marker

    ) in (

        (
            "exact",
            "Exact-grid $N$",
            "#555555",
            "o"
        ),

        (
            "interpolation",
            "Interpolated $N$",
            "#0072B2",
            "D"
        )

    ):

        R,m=curve(
            split,
            key
        )


        ok=np.isfinite(
            m
        )


        ax.plot(

            R[
                ok
            ],

            np.maximum(
                m[
                    ok
                ],
                1e-12
            ),

            color=color,

            marker=marker,

            lw=2,

            label=label

        )


    ax.set_xscale(
        "log"
    )


    ax.set_yscale(
        "log"
    )


    ax.set_xticks(
        R_VALUES
    )


    ax.set_xticklabels(
        [
            str(R)
            for R in R_VALUES
        ]
    )


    ax.set_xlabel(
        r"Exact training configurations $R$"
    )


    ax.set_ylabel(
        ylabel
    )


    ax.set_title(
        title
    )


    ax.grid(
        alpha=.15
    )


    ax.legend(
        frameon=False
    )


fig.suptitle(

    "Accuracy Versus Exact-Teacher Training-Set Size",

    fontsize=15

)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "figure_5_1B_learning_curves_R100000.pdf",

    bbox_inches="tight"

)


plt.savefig(

    OUT
    /
    "figure_5_1B_learning_curves_R100000.png",

    dpi=cfg.dpi,

    bbox_inches="tight"

)


plt.show()


# =====================================================================================
# 41. FIGURE 2 — i0=1 VS i0>1
# =====================================================================================

fig,axs=plt.subplots(

    2,
    3,

    figsize=(
        17,
        9
    )

)


spec2=[

    (
        "E2",
        r"$E_2$",
        "(A) Distributional error"
    ),

    (
        "E_rho",
        r"$E_\rho$",
        "(B) Tail-risk error"
    ),

    (
        "E_overflow",
        "Overflow error",
        "(C) Overflow risk"
    ),

    (
        "KL",
        "KL divergence",
        "(D) KL divergence"
    ),

    (
        "mean_tau_relative_error",
        "Relative error",
        r"(E) $E(\tau)$"
    ),

    (
        "var_tau_relative_error",
        "Relative error",
        r"(F) $\mathrm{Var}(\tau)$"
    )

]


for ax,(
    key,
    ylabel,
    title

) in zip(
    axs.flat,
    spec2
):

    for (
        split,
        label,
        color,
        marker

    ) in (

        (
            "interpolation_i0_1",
            r"$i_0=1$",
            "#0072B2",
            "o"
        ),

        (
            "interpolation_i0_gt1",
            r"$i_0>1$",
            "#E69F00",
            "s"
        )

    ):

        R,m=curve(
            split,
            key
        )


        ok=np.isfinite(
            m
        )


        ax.plot(

            R[
                ok
            ],

            np.maximum(
                m[
                    ok
                ],
                1e-12
            ),

            color=color,

            marker=marker,

            lw=2,

            label=label

        )


    ax.set_xscale(
        "log"
    )


    ax.set_yscale(
        "log"
    )


    ax.set_xticks(
        R_VALUES
    )


    ax.set_xticklabels(
        [
            str(R)
            for R in R_VALUES
        ]
    )


    ax.set_xlabel(
        r"Exact training configurations $R$"
    )


    ax.set_ylabel(
        ylabel
    )


    ax.set_title(
        title
    )


    ax.grid(
        alpha=.15
    )


    ax.legend(
        frameon=False
    )


fig.suptitle(

    r"Interpolation Accuracy for $i_0=1$ and $i_0>1$",

    fontsize=15

)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "figure_5_1B_i0_learning_curves_R100000.pdf",

    bbox_inches="tight"

)


plt.savefig(

    OUT
    /
    "figure_5_1B_i0_learning_curves_R100000.png",

    dpi=cfg.dpi,

    bbox_inches="tight"

)


plt.show()


# =====================================================================================
# 42. FIGURE 3 — PMF RECONSTRUCTION
# =====================================================================================

available_gallery_R=[

    R

    for R in GALLERY_R

    if all(
        R in gallery[
            label
        ]
        for label,_ in SHOWCASES
    )

]


if available_gallery_R:

    fig,axs=plt.subplots(

        2,

        len(
            available_gallery_R
        ),

        figsize=(
            5.7
            *
            len(
                available_gallery_R
            ),
            8
        ),

        squeeze=False

    )


    for row,(
        label,
        _
    ) in enumerate(
        SHOWCASES
    ):

        for col,R in enumerate(
            available_gallery_R
        ):

            ax=axs[
                row,
                col
            ]


            z=gallery[
                label
            ][R]


            c=np.arange(
                z["N"]+1
            )


            ax.bar(

                c,

                z[
                    "exact_p"
                ][:-1],

                width=.85,

                color=".82",

                label="Exact Markovian"

            )


            ax.plot(

                c,

                z[
                    "pred_p"
                ][:-1],

                color="#D55E00",

                lw=1.8,

                label="Neural emulator"

            )


            ax.set_title(

                f"{label}, $R={R:,}$\n"
                f"$N={z['N']}$, "
                f"$i_0={z['i0']}$, "
                rf"$R_0={z['R0']:.2f}$, "
                rf"$E_2={z['E2']:.3f}$, "
                rf"$E_\rho={z['E_rho']:.3f}$"

            )


            ax.set_xlabel(
                "Infection count $c$"
            )


            ax.set_ylabel(
                "Probability mass"
            )


            if (
                row==0
                and
                col==0
            ):

                ax.legend(
                    frameon=False
                )


    plt.tight_layout()


    plt.savefig(

        OUT
        /
        "figure_5_1B_pmf_vs_R100000.pdf",

        bbox_inches="tight"

    )


    plt.savefig(

        OUT
        /
        "figure_5_1B_pmf_vs_R100000.png",

        dpi=cfg.dpi,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================================
# 43. FIGURE 4 — TAIL RECONSTRUCTION
# =====================================================================================

if available_gallery_R:

    fig,axs=plt.subplots(

        2,

        len(
            available_gallery_R
        ),

        figsize=(
            5.7
            *
            len(
                available_gallery_R
            ),
            8
        ),

        squeeze=False

    )


    for row,(
        label,
        _
    ) in enumerate(
        SHOWCASES
    ):

        for col,R in enumerate(
            available_gallery_R
        ):

            ax=axs[
                row,
                col
            ]


            z=gallery[
                label
            ][R]


            c=np.arange(
                z["N"]+1
            )


            ax.plot(

                c,

                z[
                    "exact_tail"
                ],

                color="black",

                lw=2,

                label="Exact Markovian"

            )


            ax.plot(

                c,

                z[
                    "pred_tail"
                ],

                "--",

                color="#0072B2",

                lw=1.8,

                label="Neural emulator"

            )


            ax.set_ylim(
                -.01,
                1.01
            )


            ax.set_title(

                f"{label}, $R={R:,}$\n"
                f"$N={z['N']}$, "
                f"$i_0={z['i0']}$, "
                rf"$R_0={z['R0']:.2f}$, "
                rf"$E_\rho={z['E_rho']:.3f}$"

            )


            ax.set_xlabel(
                "Threshold $c$"
            )


            ax.set_ylabel(
                r"$P(C>c)$"
            )


            if (
                row==0
                and
                col==0
            ):

                ax.legend(
                    frameon=False
                )


    plt.tight_layout()


    plt.savefig(

        OUT
        /
        "figure_5_1B_tail_vs_R100000.pdf",

        bbox_inches="tight"

    )


    plt.savefig(

        OUT
        /
        "figure_5_1B_tail_vs_R100000.png",

        dpi=cfg.dpi,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================================
# 44. VALIDATION-LOSS CURVES
#
# Useful because all histories are now permanently saved.
# =====================================================================================

plt.figure(
    figsize=(8,5)
)


for R in R_VALUES:

    if R not in progress[
        "results"
    ]:

        continue


    history=progress[
        "results"
    ][R].get(
        "history",
        {}
    )


    val=np.asarray(
        history.get(
            "validation",
            []
        ),
        dtype=float
    )


    if len(val)==0:

        continue


    ep=np.arange(
        1,
        len(val)+1
    )


    plt.plot(

        ep,

        val,

        lw=1.8,

        label=rf"$R={R:,}$"

    )


plt.yscale(
    "log"
)


plt.xlabel(
    "Epoch"
)


plt.ylabel(
    "Validation loss"
)


plt.title(
    "Validation-loss trajectories"
)


plt.grid(
    alpha=.15
)


plt.legend(
    frameon=False
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "figure_5_1B_validation_losses_R100000.pdf",

    bbox_inches="tight"

)


plt.savefig(

    OUT
    /
    "figure_5_1B_validation_losses_R100000.png",

    dpi=cfg.dpi,

    bbox_inches="tight"

)


plt.show()


# =====================================================================================
# 45. FINAL RESULT FILE
# =====================================================================================

result_path=(

    OUT

    /

    "section_5_1B_R100000_single_rep_results.pkl"

)


atomic_pickle(

    {

        "config":
            asdict(
                cfg
            ),

        "R_VALUES":
            R_VALUES,

        "REPLICATIONS":
            1,

        "EXACT_TEST_N":
            EXACT_TEST_N,

        "INTERP_TEST_N":
            INTERP_TEST_N,

        "nested_design":{

            "N_BASE":
                N_BASE,

            "N_EXTRA":
                N_EXTRA,

            "base_seed":
                cfg.seed+1,

            "extra_seed":
                cfg.seed+1001,

            "base_order_seed":
                cfg.seed+20000,

            "extra_order_seed":
                cfg.seed+21000

        },

        "rows":
            rows,

        "gallery":
            gallery,

        "tau_resolution":{

            "train_base":
                (
                    sum(
                        r.tau_valid
                        for r in train_base
                    ),
                    len(
                        train_base
                    )
                ),

            "train_extra":
                (
                    sum(
                        r.tau_valid
                        for r in train_extra
                    ),
                    len(
                        train_extra
                    )
                ),

            "train_total":
                (
                    sum(
                        r.tau_valid
                        for r in train
                    ),
                    len(
                        train
                    )
                ),

            "validation":
                (
                    sum(
                        r.tau_valid
                        for r in valid
                    ),
                    len(
                        valid
                    )
                ),

            "test_exact":
                (
                    sum(
                        r.tau_valid
                        for r in test_seen
                    ),
                    len(
                        test_seen
                    )
                ),

            "test_interp":
                (
                    sum(
                        r.tau_valid
                        for r in test_interp
                    ),
                    len(
                        test_interp
                    )
                )

        }

    },

    result_path

)


# =====================================================================================
# 46. FINAL STATUS
# =====================================================================================

print(
    "\n"
    +
    "="*115
)


print(
    "EXPERIMENT 5.1-B STATUS"
)


print(
    "="*115
)


print(
    "Persistent Google Drive root:"
)


print(
    ROOT
)


print()


print(
    "Original training records:",
    f"{len(train_base):,}"
)


print(
    "Additional training records:",
    f"{len(train_extra):,}"
)


print(
    "Total training records:",
    f"{len(train):,}"
)


print()


print(
    "Original 50k cache verified:",
    OLD_BASE_VERIFIED
)


print(
    "Nested preservation:",
    "20k ⊂ 30k ⊂ 50k ⊂ 100k"
)


print()


for R in R_VALUES:

    print(
        f"R={R:6d}:",
        (
            "DONE"
            if R in progress[
                "results"
            ]
            else
            "NOT YET"
        )
    )


print()


print(
    "Exact-target chunk size:",
    EXACT_CHUNK
)


print(
    "Mid-epoch neural checkpoint:",
    f"every {BATCH_CHECKPOINT_EVERY} mini-batches"
)


print(
    "Batch size:",
    cfg.batch_size
)


print(
    "Approximate batches/epoch at R=100k:",
    math.ceil(
        100000
        /
        cfg.batch_size
    )
)


print(
    "Device:",
    device
)


print()


print(
    "Summary CSV:"
)

print(
    OUT
    /
    "section_5_1B_R100000_summary.csv"
)


print(
    "Summary LaTeX:"
)

print(
    OUT
    /
    "section_5_1B_R100000_summary.tex"
)


print(
    "Final result pickle:"
)

print(
    result_path
)


print()


print(
    "AFTER A GOOGLE COLAB DISCONNECT:"
)


print(
    "1. Reconnect."
)


print(
    "2. Run THIS SAME CELL."
)


print(
    "3. Previously completed exact chunks are skipped."
)


print(
    "4. The original 50k targets are not recomputed if verified."
)


print(
    "5. Completed R=20k/30k/50k results are reused when compatible."
)


print(
    "6. An interrupted R=100k epoch resumes from the latest 50-batch checkpoint."
)


print(
    "7. Completed R values are skipped completely."
)


print(
    "8. All persistent files remain in Google Drive."
)


print(
    "="*115
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXPERIMENT 5.1-B — R UP TO 100,000
Permanent root: /content/drive/MyDrive/StatisticalLearning/Experiment_5_1B_R100000_resumable_v16
Previous 50k root: /content/drive/MyDrive/StatisticalLearning/Experiment_5_1B_R50000_resumable_v15
R values: (20000, 30000, 50000, 100000)
Base configurations: 50,000
Additional configurations: 50,000
Total configurations: 100,000
Device: cpu
Logical CPU cores: 2
Exact workers: 2

No verified previous 50k training cache found.
The deterministic original 50k design will be generated.

train_base_R50000: 50,000 exact targets
